# Zolai LLM Fine-Tuning — 3B Model (Quality)
Model: **Qwen2.5-3B-Instruct** | 100k samples/session | Optimized for quality

**Each session:** update `CHUNK_START` and `RESUME_ADAPTER` in the Session Config cell.

| Session | Chunk | Status |
|---------|-------|--------|
| 1 | 0 → 25,000 | done |
| 2 | 25,000 → 50,000 | done |
| 3 | 50,000 → 75,000 | done |
| 4 | 75,000 → 175,000 | ← next |

## 1. Install Dependencies

In [ ]:
!pip install -q --upgrade torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==5.5.4 peft==0.19.1 trl==1.2.0 datasets accelerate==1.13.0 bitsandbytes==0.49.2 kaggle huggingface_hub --upgrade


## 2. Environment & Secrets

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ["HF_TOKEN"]       = _s.get_secret("HF_TOKEN")
    os.environ["KAGGLE_USERNAME"] = "peterpausianlian"
    os.environ["KAGGLE_KEY"]     = _s.get_secret("KAGGLE_KEY")
    print("Secrets loaded from Kaggle")
except Exception as e:
    print(f"Kaggle Secrets unavailable ({e})")

# Single GPU for 3B (4-bit quantization)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

## 3. Verify Environment

In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes

print(f"torch:         {torch.__version__}")
print(f"transformers:  {transformers.__version__}")
print(f"peft:          {peft.__version__}")
print(f"accelerate:    {accelerate.__version__}")
print(f"bitsandbytes:  {bitsandbytes.__version__}")

n = torch.cuda.device_count()
print(f"\nGPUs visible: {n} (single-GPU mode)")
for i in range(n):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | {free/1e9:.1f}GB free / {total/1e9:.1f}GB total")
assert n >= 1, "No GPU found!"
print("\n✓ All checks passed")

## 4. Session Config
> **Update `CHUNK_START` and `RESUME_ADAPTER` every session.**

In [ ]:
# ── UPDATE EACH SESSION ──────────────────────────────────────────
CHUNK_START    = 300000                                      # 0, 300000, 800000 ...
RESUME_ADAPTER = "peterpausianlian/zolai-qwen2.5-3b-lora"   # None for first run only
# ─────────────────────────────────────────────────────────────────

CHUNK_SIZE  = 500_000
MAX_VAL     = 500
MAX_LENGTH  = 128
BATCH_SIZE  = 4
MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
HF_REPO     = "peterpausianlian/zolai-qwen2.5-3b-lora"
KAGGLE_DS   = "peterpausianlian/zolai-adapter-qwen25-3b"

print(f"Session: chunk {CHUNK_START:,} → {CHUNK_START + CHUNK_SIZE:,}")
print(f"Resume:  {RESUME_ADAPTER or 'None (fresh start)'}")


## 5. Load Dataset

In [ ]:
import json, glob
from datasets import Dataset

train_files = glob.glob("/kaggle/input/**/llm_train.jsonl", recursive=True)
val_files   = glob.glob("/kaggle/input/**/llm_val.jsonl",   recursive=True)

if not train_files:
    print("ERROR: Dataset not found. Available jsonl:")
    for f in glob.glob("/kaggle/input/**/*.jsonl", recursive=True): print(f"  {f}")
    raise FileNotFoundError("llm_train.jsonl not found in /kaggle/input")

TRAIN_FILE = train_files[0]
VAL_FILE   = val_files[0] if val_files else train_files[0]
print(f"Train: {TRAIN_FILE}")
print(f"Val:   {VAL_FILE}")

# Count total lines to validate chunk bounds
with open(TRAIN_FILE) as f:
    TOTAL_TRAIN = sum(1 for _ in f)
print(f"Total train rows: {TOTAL_TRAIN:,}")
if CHUNK_START >= TOTAL_TRAIN:
    raise ValueError(f"CHUNK_START={CHUNK_START:,} exceeds dataset size {TOTAL_TRAIN:,}. Training complete!")
actual_chunk = min(CHUNK_SIZE, TOTAL_TRAIN - CHUNK_START)
if actual_chunk < CHUNK_SIZE:
    print(f"⚠ Last chunk: only {actual_chunk:,} rows available (dataset end)")

def load_jsonl(filepath, offset=0, limit=None):
    texts = []
    with open(filepath) as f:
        for i, line in enumerate(f):
            if i < offset: continue
            if limit and i >= offset + limit: break
            t = json.loads(line).get("text", "").strip()
            if t: texts.append(t)
    return Dataset.from_dict({"text": texts})

train_dataset = load_jsonl(TRAIN_FILE, offset=CHUNK_START, limit=CHUNK_SIZE)
val_dataset   = load_jsonl(VAL_FILE, limit=MAX_VAL)
print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")

## 6. Load Model (4-bit QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, PeftModel, LoraConfig, TaskType
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

HF_TOKEN = os.environ.get("HF_TOKEN")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

# Use "auto" for dual GPU, "cuda:0" for single GPU (4-bit requires single)
device_map = "cuda:0" if torch.cuda.device_count() == 1 else "auto"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map=device_map,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

if RESUME_ADAPTER:
    print(f"Resuming from: {RESUME_ADAPTER}")
    model = PeftModel.from_pretrained(base_model, RESUME_ADAPTER, is_trainable=True, token=HF_TOKEN)
else:
    print("Fresh LoRA init")
    model = get_peft_model(base_model, LoraConfig(
        r=8, lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))

model.print_trainable_parameters()
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {(total-free)/1e9:.1f}GB used / {total/1e9:.1f}GB total")
print(f"✓ Model loaded on {torch.cuda.device_count()} GPU(s)")

## 7. Verify Model (Quick Test)

In [ ]:
test = tokenizer("Kei ka min Peter hi.", return_tensors="pt").to("cuda:0")
with torch.no_grad():
    out = model.generate(**test, max_new_tokens=20, do_sample=False)
print("Test:", tokenizer.decode(out[0], skip_special_tokens=True))
print(f"Adapter: {RESUME_ADAPTER or 'None (fresh)'}")

## 8. Tokenize & Setup Trainer

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

print("Tokenizing...")
train_tok = train_dataset.map(tokenize, batched=True, remove_columns=["text"], num_proc=2)
val_tok   = val_dataset.map(tokenize,   batched=True, remove_columns=["text"], num_proc=2)

training_args = TrainingArguments(
    output_dir="./zolai_model",
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=8,
    warmup_steps=100,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    max_grad_norm=0.3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

steps = len(train_tok) // (BATCH_SIZE * 8)
print(f"Ready | Train: {len(train_tok):,} | Steps: {steps:,} | ~{steps*11/3600:.1f}h")

## 9. Train

In [ ]:
import datetime
_start = datetime.datetime.now()
print(f"Starting training... [{_start.strftime('%H:%M:%S')}]")
trainer.train()
_elapsed = datetime.datetime.now() - _start
print(f"✓ Training complete! Duration: {str(_elapsed).split('.')[0]}")

# Print eval checkpoints + final steps
print("\nEval checkpoints:")
print(f"{'Step':>6} | {'Train Loss':>10} | {'Val Loss':>9}")
print("-" * 34)
for l in trainer.state.log_history:
    if "eval_loss" in l:
        step = l.get("step", "?")
        # find nearest train loss
        train_loss = next((x["loss"] for x in trainer.state.log_history
                           if x.get("step") == step and "loss" in x), None)
        print(f"{step:>6} | {train_loss or '':>10} | {l['eval_loss']:>9.6f}")

print("\nFinal 10 training steps:")
print(f"{'Step':>6} | {'Train Loss':>10}")
print("-" * 22)
train_logs = [l for l in trainer.state.log_history if "loss" in l and "eval_loss" not in l]
for l in train_logs[-10:]:
    print(f"{l['step']:>6} | {l['loss']:>10.4f}")

## 10. Save & Upload Adapter
> Uploads to HF Hub and Kaggle. Downloads automatically next session via `RESUME_ADAPTER`.

In [ ]:
import os
import shutil
import json as _json
import subprocess
from datetime import datetime
from huggingface_hub import HfApi, create_repo

# ========= CONFIG =========
ADAPTER_OUT = "./zolai_adapter"
SAVE_TOKENIZER = False
VERSION = f"chunk-{CHUNK_START}-{CHUNK_START + CHUNK_SIZE}"
COMMIT_MSG = f"{VERSION} | Zolai training"

# ========= SAVE MODEL =========
if os.path.exists(ADAPTER_OUT):
    shutil.rmtree(ADAPTER_OUT)

os.makedirs(ADAPTER_OUT, exist_ok=True)
model.save_pretrained(ADAPTER_OUT)

if SAVE_TOKENIZER:
    tokenizer.save_pretrained(ADAPTER_OUT)

# ========= ZIP =========
zip_path = shutil.make_archive("zolai_adapter", "zip", ".", ADAPTER_OUT)
print(f"Zipped: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

# ========= HUGGING FACE =========
token = os.environ["HF_TOKEN"]
api = HfApi()

create_repo(HF_REPO, repo_type="model", token=token, exist_ok=True, private=True)

api.upload_folder(
    folder_path=ADAPTER_OUT,
    repo_id=HF_REPO,
    repo_type="model",
    token=token,
    commit_message=COMMIT_MSG,
    delete_patterns=["*"]
)

print(f"HF Hub: https://huggingface.co/{HF_REPO}")

# ========= KAGGLE =========
os.makedirs("kaggle_upload", exist_ok=True)

for f in os.listdir("kaggle_upload"):
    os.remove(os.path.join("kaggle_upload", f))

shutil.copy(zip_path, "kaggle_upload/zolai_adapter.zip")

with open("kaggle_upload/dataset-metadata.json", "w") as f:
    _json.dump({
        "title": "zolai-adapter-qwen25-3b",
        "id": KAGGLE_DS,
        "licenses": [{"name": "MIT"}]
    }, f)

result = subprocess.run(
    ["kaggle", "datasets", "version", "-p", "kaggle_upload", "-m", COMMIT_MSG, "--dir-mode", "zip"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"Kaggle: https://www.kaggle.com/datasets/{KAGGLE_DS}")
else:
    print(f"Kaggle upload failed: {result.stderr}")

# ========= NEXT SESSION =========
print(f"\n{'='*50}")
print(f"Next session config:")
print(f"CHUNK_START = {CHUNK_START + CHUNK_SIZE}")
print(f'RESUME_ADAPTER = "{HF_REPO}"')
print(f"{'='*50}")